# 10 — Experiment Runner (Stage 2)

Thin launcher over `prjudge.judge`. All logic lives in the package; this notebook only configures, launches, and monitors runs. Resumability lives in the JSONL checkpoint — a dead kernel costs nothing, just re-run the cell.

**Workflow:** config sanity → pilot (tune the prompt) → dry-run (real APIs, cheap) → final battery. Pilot output is never mixed with `results_v1.jsonl`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))  # run from notebooks/
from prjudge.config import load_config
config = load_config()
from prjudge.judge import resolve_run_spec, run_judges
from prjudge.batch import run_judges_batch
print('config hash:', config.hash()[:16])
print('judges:', [j['name']+' -> '+j['model'] for j in config['judging']['judges']])
print('trials:', config['judging']['trials'], '| prompt:', config['judging']['prompt_version'])

## Config sanity
Confirm the selection manifest and variants are frozen before judging.

In [ ]:
import json
man = json.load(open(config.artifacts_dir / 'selection_manifest_v1.json'))
print('selected PRs:', man['n_selected'], '| composition:', man['composition'])
vdir = config.artifacts_dir / 'variants_v1'
print('variant files:', len(list(vdir.glob('*__*.json'))) if vdir.exists() else 'MISSING — run 01_build_variants.py')

## Pilot cell — tune the system prompt
Run *baseline × 1 judge × a few PRs* repeatedly while iterating on the prompt in `prjudge.prompts` (bump to `v2`, `v3`, …). Each `run_name` writes its own file. Use `mock=True` for a no-spend plumbing check first.

In [ ]:
pilot = resolve_run_spec(config, run_name='pilot_v1',
                         variants=['baseline'], judges=['claude'], limit=5,
                         trials=3, prompt_version='v1')
summary = run_judges(config, pilot, mock=True)   # flip to mock=False to spend
print(summary)

### Inspect parsed checklist outputs side-by-side
Judge the prompt quality: are the answers grounded, is the evidence sane?

In [ ]:
import json, pandas as pd
rows = [json.loads(l) for l in open(config.artifacts_dir / 'runs' / 'pilot_v1.jsonl')]
view = []
for r in rows[:6]:
    p = r.get('parsed') or {}
    cl = p.get('checklist', {})
    view.append({'task': r['task_id'], 'trial': r['trial'],
                 **{f'i{i}': cl.get(f'item_{i}', {}).get('answer') for i in range(1, 11)},
                 'justif': (p.get('justification') or '')[:60]})
pd.DataFrame(view)

## Dry-run — validate schema enforcement on real APIs (~$0.10)
2 PRs × 1 variant × 1 judge × 1 trial. Confirms structured output parses on the pinned provider before committing to the full battery.

In [ ]:
dry = resolve_run_spec(config, run_name='dry_run', variants=['baseline'],
                       judges=[config['judging']['judges'][0]['name']], limit=2, trials=1)
# run_judges(config, dry, mock=False)   # uncomment to spend ~$0.10
print('dry-run spec:', len(dry.cells()), 'cells')

## Final battery — full matrix, frozen prompt
40 × 9 × 2 × 3 = 2,160 calls → `artifacts/runs/results_v1.jsonl`. Resumable: re-run this cell after any interruption and it continues from the checkpoint.

In [ ]:
final = resolve_run_spec(config, run_name=config['judging']['final_run_name'])
print('final spec:', len(final.cells()), 'cells,', 'prompt', final.prompt_version)
# summary = run_judges(config, final, mock=False)   # uncomment for the real run
# print(summary)

## Final run (batch) — 50% cheaper, same models

Alternative to the cell above for the final battery (or any large extension-pool
run): submits/collects through each provider's Batch API instead of calling
synchronously. `run_judges_batch` is an idempotent *advance* step — re-run this
cell later to collect finished provider batches and submit whatever is still
missing; it reports `collected_now` / `in_flight` / `needs_sync` each time.
Writes to the exact same `{run_name}.jsonl` as the sync cell above, so the two
can be freely mixed (each row's `api_mode` records which path produced it).

In [ ]:
final = resolve_run_spec(config, run_name=config['judging']['final_run_name'])
print('final spec:', len(final.cells()), 'cells,', 'prompt', final.prompt_version)
# batch_summary = run_judges_batch(config, final, mock=False)   # uncomment; re-run cell later to collect
# print(batch_summary)

## Progress / cost summary

In [ ]:
from prjudge.judge import existing_keys
run = config['judging']['final_run_name']
path = config.artifacts_dir / 'runs' / f'{run}.jsonl'
done = existing_keys(path) if path.exists() else set()
print(f'{run}: {len(done)} / {len(final.cells())} cells complete')